# Advanced usage: The `MorphingWorkflow` classes: `MorphingWorkflowGlobal`, `MorphingWorkflowEurope` and `MorphingWorkflowCordex`

This notebook demonstrates the advanced, step-by-step approach to morphing EPW files using the specialized `MorphingWorkflow` classes (`MorphingWorkflowGlobal`, `MorphingWorkflowEurope` and `MorphingWorkflowCordex`).

While the `morph_epw_*` functions are great for direct, one-shot tasks, the workflow classes are designed for complex projects where you need full control over **filename parsing, custom renaming, and process validation**. They enforce a safe, multi-step process that allows you to review and confirm each stage before executing the time-consuming morphing computation.

This notebook will focus on **`MorphingWorkflowGlobal`**, and will show quick examples using **`MorphingWorkflowEurope`** and **`MorphingWorkflowCordex`** highlighting the differences.

**Note:** `pyfwg` fully supports **FutureWeatherGenerator v4.0.x** (Global) and **v2.0.x** (Europe), which include a new keyword-based CLI, support for string-based parameter options, and additional output formats (like .met or .csv).

**New in pyfwg v0.4.0:** `pyfwg` also supports the **FWG Engine**, the shared engine of **Global v5.x**, **Europe v3.x** and the **CORDEX regional** distributions. `MorphingWorkflowGlobal` and `MorphingWorkflowEurope` work with Global v5.x and Europe v3.x, and the new `MorphingWorkflowCordex` class handles the CORDEX regional distributions.

## Configuration

The examples in this notebook run the Future Weather Generator, which is not distributed with `pyfwg`. Download its JAR files from the [official website](https://future-weather-generator.adai.pt/) and put them in a folder. Then, set the `FWG_JAR_DIR` environment variable to that folder before starting Jupyter, or replace `'path/to/jars'` with its path in the cell below. All the JAR paths of this notebook are built from `FWG_JAR_DIR`: if you use other versions of the tool, change the filenames where the paths are defined.

In the saved outputs, `<FWG_JAR_DIR>`, `<TEMP>` and `<HOME>` stand for the JAR folder, the temporary folder and the home folder of the computer where the notebook was run.

In [ ]:
import os

# Folder that contains the JAR files of the Future Weather Generator. Set the FWG_JAR_DIR
# environment variable, or replace 'path/to/jars' with the folder on your computer (keep the
# r prefix, so that the backslashes of a Windows path are not read as escape sequences).
FWG_JAR_DIR = os.environ.get('FWG_JAR_DIR', r'path/to/jars')

## The Three-Step Workflow

The class is designed to be used as a state machine, guiding you through a logical sequence:

1.  **`map_categories()`**: Analyze the input filenames to extract meaningful data (like city, building type, etc.).
2.  **`configure_and_preview()`**: Define all execution parameters, validate them, and generate a "dry run" plan of how the final files will be named and organized.
3.  **`execute_morphing()`**: Run the final computation, confident that your plan and configuration are correct.

## Using `MorphingWorkflowGlobal`

### General Setup

First, we'll import the necessary class and set up the paths for our files. 

**Important:** `jar_path` is built from `FWG_JAR_DIR`, set in the *Configuration* section at the top of the notebook. If your Future Weather Generator `.jar` file has a different name, change it in the cell below. `pyfwg` will automatically detect the version and adapt the CLI command accordingly.

In [15]:
import os
import pandas as pd
from pyfwg import MorphingWorkflowGlobal, DEFAULT_GLOBAL_GCMS

# --- Configuration ---
# V4 JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename if you use another version.
jar_path = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator_v4.0.2.jar')

# --- Define file paths for the examples ---
pattern_epw_dir = 'epws/w_pattern'
keyword_epw_dir = 'epws/wo_pattern'

### Step 1: Map Categories from Filenames

This is the first and most critical step for organizing your workflow. The `map_categories` method analyzes your source filenames and extracts meaningful data that will be used later for renaming the output files.

#### `map_categories` Parameters

The parameters you can use are:

*   `epw_files` (`List[str]`): **Required.** A list of paths to the EPW files you want to process.
*   `input_filename_pattern` (`Optional[str]`, default: `None`): A Python regex string with **named capture groups** (e.g., `(?P<city>...)`) to extract structured data from filenames.
*   `keyword_mapping` (`Optional[Dict]`, default: `None`): A dictionary of rules. It can be used to normalize values from the pattern or to search the entire filename for keywords to assign categories.

In [16]:
# Instantiate a workflow object for this example
workflow = MorphingWorkflowGlobal()

epw_files_without_pattern = [os.path.join(keyword_epw_dir, f) for f in os.listdir(keyword_epw_dir) if f.endswith('.epw')]

mapping_rules = {
    'city': {
        'Seville': ['sevilla', 'SVQ'],
        'London': ['london', 'gatwick']
    },
    'uhi': {
        'type-1': 'type-1',
        'type-2': 'type-2'
    }
}

workflow.map_categories(
    epw_files=epw_files_without_pattern,
    input_filename_pattern=None,
    keyword_mapping=mapping_rules
)

2025-12-30 12:39:49 - INFO - --- Step 1: Mapping categories from filenames ---
2025-12-30 12:39:49 - INFO - Mapped 'epws/wo_pattern\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw': {'city': 'London', 'uhi': 'type-2'}
2025-12-30 12:39:49 - INFO - Mapped 'epws/wo_pattern\sevilla_in_this_one_the_uhi_is_type-1.epw': {'city': 'Seville', 'uhi': 'type-1'}
2025-12-30 12:39:49 - INFO - Category mapping complete.


### Step 2: Configure and Preview the Plan (V4 Update)

In V4, you can use descriptive strings for interpolation methods (e.g., `'AVG4P'`) and solar hour adjustments (e.g., `'By_Month'`). You can also specify the `fwg_version` to force the new CLI.

#### `configure_and_preview` Parameters

##### Workflow Parameters
*   `final_output_dir` (`str`): **Required.** The path for the final output files.
*   `output_filename_pattern` (`str`): **Required.** The template for final filenames.
*   `scenario_mapping` (`Optional[Dict]`, default: `None`): A dictionary to translate raw scenario names.
*   `fwg_jar_path` (`str`): **Required.** Path to the `.jar` file.
*   `fwg_version` (`Optional[str]`): Force a specific CLI version (e.g., `'4'`).
*   `fwg_output_type` (`str`, default: `'EPW'`): **New in V4/Europe v2.** Format of output files: `'EPW'`, `'SPAIN_MET'`, or `'PORTUGAL_CSV'`.

##### FWG Engine Parameters (Global v5.x, Europe v3.x and CORDEX regional)
*   `fwg_scenarios` and `fwg_timeframes` (`Optional[List]`, default: `None`): Subsets of scenarios (e.g., `['ssp245']`) and future periods (e.g., `[2050]`) to generate. The preview only includes the selected ones.
*   `fwg_uncertainty_variant` and `fwg_uncertainty_sigma` (default: `None`): Uncertainty case of the climate-change signal (e.g., `'CENTRAL'` or `'WARM_DRY'`) and its standard-deviation multiplier.
*   `fwg_decomposition_method`, `fwg_illuminance_method`, `fwg_infrared_method` and `fwg_morphing_method` (default: `None`): Models used by the FWG Engine.
*   `fwg_output_profile` (`Optional[str]`, default: `None`): Reports written by the tool: `'MINIMAL'` (weather files only), `'STANDARD'`, `'FULL'` or `'CUSTOM'`.
*   `fwg_model_unzipped_folder` (`Optional[str]`, default: `None`): Folder to cache the unzipped climate-model data across runs.
*   `fwg_extra_args` (`Optional[Dict]`, default: `None`): Any other option of the tool (e.g., `{'threads': 4}`). Also available for Global v4.x and Europe v2.x.

Arguments left as `None` keep the default of the tool. Like any other FWG parameter, they can be used as placeholders in `output_filename_pattern` (e.g., `{fwg_uncertainty_variant}`). See the Global v5.x example below.

In [17]:
workflow.configure_and_preview(
    final_output_dir='./final_results_workflow_global',
    output_filename_pattern='{city}_{uhi}_{ssp}_{year}_interp-{fwg_interpolation_method_id}',
    scenario_mapping={'ssp245': 'SSP2-4.5', 'ssp585': 'SSP5-8.5'},
    
    # --- FWG Configuration ---
    fwg_jar_path=jar_path,
    fwg_gcms=['BCC_CSM2_MR'], 
    fwg_interpolation_method_id='AVG4P', # V4: Descriptive string
    fwg_epw_original_lcz=2,        
    fwg_target_uhi_lcz=3,          
    fwg_version='4',                # Force V4 syntax
    fwg_show_tool_output=True
)

2025-12-30 12:39:49 - INFO - --- Step 2: Configuring and Previewing Morphing Plan ---



          MORPHING CONFIGURATION & PREVIEW
  - FWG JAR Path: <FWG_JAR_DIR>\FutureWeatherGenerator_v4.0.2.jar
  - Final Output Directory: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_global
  - EPWs to be Morphed (2 files):
    - GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw
    - sevilla_in_this_one_the_uhi_is_type-1.epw

  For input file: GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw
    -> Generated 'ssp126_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_global\London_type-2_ssp126_2050_interp-AVG4P.epw
    -> Generated 'ssp245_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_global\London_type-2_SSP2-4.5_2050_interp-AVG4P.epw
    -> Generated 'ssp370_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_global\London_type-2_ssp370_2050_interp-AVG4P.epw
    -> Generated 'ssp585_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_global\London_typ

### Step 3: Execute the Morphing Process

This is the final step. The `execute_morphing` method runs the computation. With V4, it will use the new keyword-based CLI.

In [18]:
workflow.execute_morphing()

2025-12-30 12:39:49 - INFO - --- Step 4: Executing morphing workflow ---
2025-12-30 12:39:49 - INFO - Validating LCZ availability for GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw...
2025-12-30 12:39:49 - INFO - Checking LCZ pair (Original: 2, Target: 3) availability for GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw...
2025-12-30 12:39:49 - INFO - --- Applying UHI effect to GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw ---
2025-12-30 12:39:49 - INFO - Executing command: java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_v4.0.2.jar" -u -epw=<TEMP>\tmp9ikq7m2w\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw -output_folder=<TEMP>\tmp9ikq7m2w\ -uhi=true:2:3 -output_type=EPW
2025-12-30 12:40:05 - INFO - UHI effect applied successfully.
2025-12-30 12:40:05 - INFO - LCZ pair (Original: 2, Target: 3) is available.
2025-12-30 12:40:05 - INFO - Copied input file to temporary directory: ./morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw



-------------------- Executing FWG for GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw --------------------
  Full Command (for reference): java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_v4.0.2.jar" -epw=D:\Python\pyfwg\pyfwg\tutorials\morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw D:\Python\pyfwg\pyfwg\tutorials\epws\wo_pattern\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw -models=BCC_CSM2_MR -ensemble=true -temp_shift_winter=0.0 -temp_shift_summer=0.0 -smooth_hours=72 -multithread=true -grid_interpolation_method=AVG4P -solar_correction=By_Month -diffuse_method=Engerer_2015 -uhi=true:2:3 -output_type=EPW
  --- FWG Real-time Output ---


2025-12-30 12:40:32 - INFO - Processing generated files in: ./morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2
2025-12-30 12:40:32 - INFO - Skipping auxiliary file: '00_logs'
2025-12-30 12:40:32 - INFO - Skipping auxiliary file: '01_processing_transcripts'
2025-12-30 12:40:32 - INFO - Skipping auxiliary file: '02_morphing_errors_and_warnings'
2025-12-30 12:40:32 - INFO - Skipping auxiliary file: '03_model_variables'
2025-12-30 12:40:32 - INFO - Skipping auxiliary file: '04_epw_comparisons'
2025-12-30 12:40:32 - INFO - Moving './morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2\GBR_-_LONDON-GATWICK_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp126_2050.epw' to './final_results_workflow_global\London_type-2_ssp126_2050_interp-AVG4P.epw'
2025-12-30 12:40:32 - INFO - Moving './morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2\GBR_-_LONDON-GATWICK_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp126_2050.stat' to './final_results_workflow_global\London_ty

  --- End of FWG Output ---


2025-12-30 12:40:33 - WARNING - PermissionError deleting ./morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2. Retrying in 0.5s... (Attempt 2/5)
2025-12-30 12:40:33 - INFO - Validating LCZ availability for sevilla_in_this_one_the_uhi_is_type-1.epw...
2025-12-30 12:40:33 - INFO - Checking LCZ pair (Original: 2, Target: 3) availability for sevilla_in_this_one_the_uhi_is_type-1.epw...
2025-12-30 12:40:33 - INFO - --- Applying UHI effect to sevilla_in_this_one_the_uhi_is_type-1.epw ---
2025-12-30 12:40:33 - INFO - Executing command: java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_v4.0.2.jar" -u -epw=<TEMP>\tmpv9l3_d4y\sevilla_in_this_one_the_uhi_is_type-1.epw -output_folder=<TEMP>\tmpv9l3_d4y\ -uhi=true:2:3 -output_type=EPW
2025-12-30 12:40:48 - INFO - UHI effect applied successfully.
2025-12-30 12:40:48 - INFO - LCZ pair (Original: 2, Target: 3) is available.
2025-12-30 12:40:48 - INFO - Copied input file to temporary directory: ./morphing_temp_results\sevilla_in_this_one_the


-------------------- Executing FWG for sevilla_in_this_one_the_uhi_is_type-1.epw --------------------
  Full Command (for reference): java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_v4.0.2.jar" -epw=D:\Python\pyfwg\pyfwg\tutorials\morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1\sevilla_in_this_one_the_uhi_is_type-1.epw D:\Python\pyfwg\pyfwg\tutorials\epws\wo_pattern\sevilla_in_this_one_the_uhi_is_type-1.epw -models=BCC_CSM2_MR -ensemble=true -temp_shift_winter=0.0 -temp_shift_summer=0.0 -smooth_hours=72 -multithread=true -grid_interpolation_method=AVG4P -solar_correction=By_Month -diffuse_method=Engerer_2015 -uhi=true:2:3 -output_type=EPW
  --- FWG Real-time Output ---


2025-12-30 12:41:14 - INFO - Processing generated files in: ./morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1
2025-12-30 12:41:14 - INFO - Skipping auxiliary file: '00_logs'
2025-12-30 12:41:14 - INFO - Skipping auxiliary file: '01_processing_transcripts'
2025-12-30 12:41:14 - INFO - Skipping auxiliary file: '02_morphing_errors_and_warnings'
2025-12-30 12:41:14 - INFO - Skipping auxiliary file: '03_model_variables'
2025-12-30 12:41:14 - INFO - Skipping auxiliary file: '04_epw_comparisons'
2025-12-30 12:41:14 - INFO - Moving './morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp126_2050.epw' to './final_results_workflow_global\Seville_type-1_ssp126_2050_interp-AVG4P.epw'
2025-12-30 12:41:14 - INFO - Moving './morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp126_2050.stat' to './final_results_workflow_global\Seville_type-1_ssp126_2050_interp-

  --- End of FWG Output ---


2025-12-30 12:41:15 - INFO - Skipping auxiliary file: 'UHI_AirTemperature_10000'
2025-12-30 12:41:15 - INFO - Deleting temporary directory: ./morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1
2025-12-30 12:41:15 - WARNING - PermissionError deleting ./morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1. Retrying in 0.5s... (Attempt 1/5)
2025-12-30 12:41:15 - INFO - Morphing workflow finished.


Let's take a look at the files we just generated:

In [19]:
output_files = [i for i in os.listdir('./final_results_workflow_global')]
output_files

['London_type-2_ssp126_2050_interp-AVG4P.epw',
 'London_type-2_ssp126_2050_interp-AVG4P.stat',
 'London_type-2_ssp126_2080_interp-AVG4P.epw',
 'London_type-2_ssp126_2080_interp-AVG4P.stat',
 'London_type-2_SSP2-4.5_2050_interp-AVG4P.epw',
 'London_type-2_SSP2-4.5_2050_interp-AVG4P.stat',
 'London_type-2_SSP2-4.5_2080_interp-AVG4P.epw',
 'London_type-2_SSP2-4.5_2080_interp-AVG4P.stat',
 'London_type-2_ssp370_2050_interp-AVG4P.epw',
 'London_type-2_ssp370_2050_interp-AVG4P.stat',
 'London_type-2_ssp370_2080_interp-AVG4P.epw',
 'London_type-2_ssp370_2080_interp-AVG4P.stat',
 'London_type-2_SSP5-8.5_2050_interp-AVG4P.epw',
 'London_type-2_SSP5-8.5_2050_interp-AVG4P.stat',
 'London_type-2_SSP5-8.5_2080_interp-AVG4P.epw',
 'London_type-2_SSP5-8.5_2080_interp-AVG4P.stat',
 'Seville_type-1_ssp126_2050_interp-AVG4P.epw',
 'Seville_type-1_ssp126_2050_interp-AVG4P.stat',
 'Seville_type-1_ssp126_2080_interp-AVG4P.epw',
 'Seville_type-1_ssp126_2080_interp-AVG4P.stat',
 'Seville_type-1_SSP2-4.5_2050

### Global v5.x Example (FWG Engine)

`MorphingWorkflowGlobal` also works with **Global v5.x**, which runs on the FWG Engine. The three steps are the same: `pyfwg` detects the version from the JAR, validates the new options and builds the new command line for you. The main differences are:

*   You can select the scenarios and future periods to generate with `fwg_scenarios` and `fwg_timeframes`. The preview only includes the selected ones.
*   The FWG Engine does not support the Urban Heat Island effect, so the LCZ arguments are not used and the LCZ pre-flight check is skipped.
*   The new options, such as the uncertainty case of the climate-change signal (`fwg_uncertainty_variant`), can also be used as placeholders in `output_filename_pattern`.

In this example, we reuse the EPW files and the mapping rules of Step 1, and we generate a single scenario and future period with two GCMs to keep it fast. The unzipped climate-model data is cached in `model_cache_dir`, outside the project folder, so that later runs can reuse it.

**Important:** `jar_path_v5` is built from `FWG_JAR_DIR`, set in the *Configuration* section at the top of the notebook. If your Global v5 `.jar` file has a different name, change it in the cell below.

In [3]:
# Global v5 JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename if you use another version.
jar_path_v5 = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar')

# Folder to cache the unzipped climate-model data across runs. It can take several hundred MB,
# so we keep it outside the project folder (here, in the user's home folder).
model_cache_dir = os.path.join(os.path.expanduser('~'), 'fwg_model_cache', 'cmip6_global')

# Step 1: Map the categories of the same files as before
workflow_v5 = MorphingWorkflowGlobal()
workflow_v5.map_categories(
    epw_files=epw_files_without_pattern,
    keyword_mapping=mapping_rules
)

2026-10-17 22:11:48 - INFO - --- Step 1: Mapping categories from filenames ---
2026-10-17 22:11:48 - INFO - Mapped 'epws/wo_pattern\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw': {'city': 'London', 'uhi': 'type-2'}
2026-10-17 22:11:48 - INFO - Mapped 'epws/wo_pattern\sevilla_in_this_one_the_uhi_is_type-1.epw': {'city': 'Seville', 'uhi': 'type-1'}
2026-10-17 22:11:48 - INFO - Category mapping complete.


In [4]:
# Step 2: Configure and preview the plan
workflow_v5.configure_and_preview(
    final_output_dir='./final_results_workflow_global_v5',
    output_filename_pattern='{city}_{ssp}_{year}_{fwg_uncertainty_variant}',
    scenario_mapping={'ssp245': 'SSP2-4.5'},

    # --- FWG Configuration ---
    fwg_jar_path=jar_path_v5,              # The version (v5) is detected from the JAR
    fwg_gcms=['BCC_CSM2_MR', 'CanESM5'],   # At least two GCMs are needed to create the ensemble
    fwg_scenarios=['ssp245'],              # Only generate this scenario...
    fwg_timeframes=[2050],                 # ...and this future period
    fwg_uncertainty_variant='CENTRAL',     # Central climate-change signal (the default of the tool)
    fwg_output_profile='MINIMAL',          # Only write the weather files (no reports)
    fwg_model_unzipped_folder=model_cache_dir
)

2026-10-17 22:11:48 - INFO - --- Step 2: Configuring and Previewing Morphing Plan ---
2026-10-17 22:11:48 - INFO - Auto-detected FWG version: 5
2026-10-17 22:11:48 - INFO - Detected FWG CMIP6 Global v5.2.0 (FWG Engine 1.1.0).



          MORPHING CONFIGURATION & PREVIEW
  - FWG JAR Path: <FWG_JAR_DIR>\FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar
  - Final Output Directory: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_global_v5
  - EPWs to be Morphed (2 files):
    - GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw
    - sevilla_in_this_one_the_uhi_is_type-1.epw

  For input file: GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw
    -> Generated 'ssp245_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_global_v5\London_SSP2-4.5_2050_CENTRAL.epw

  For input file: sevilla_in_this_one_the_uhi_is_type-1.epw
    -> Generated 'ssp245_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_global_v5\Seville_SSP2-4.5_2050_CENTRAL.epw
Configuration set. Call execute_morphing() to start the process.


Now, we execute the morphing process. Since the FWG Engine does not support the UHI effect, the LCZs are not validated before each run.

In [5]:
# Step 3: Execute the morphing process
workflow_v5.execute_morphing()

2026-10-17 22:11:48 - INFO - --- Step 4: Executing morphing workflow ---
2026-10-17 22:11:48 - INFO - Copied input file to temporary directory: ./morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw



-------------------- Executing FWG for GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw --------------------
  Full Command (for reference): java -jar <FWG_JAR_DIR>\FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar -g -epw=D:\Python\pyfwg\pyfwg\tutorials\epws\wo_pattern\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw -output_folder=D:\Python\pyfwg\pyfwg\tutorials\morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2\ -output_layout=ORGANIZED -models=BCC_CSM2_MR,CanESM5 -ensemble=true -temp_shift_winter=0.0 -temp_shift_summer=0.0 -smooth_hours=72 -multithread=true -grid_interpolation_method=IDW -epw_dictionary_limits=true -output_type=EPW -scenarios=ssp245 -timeframes=2050 -uncertainty_variant=CENTRAL -output_profile=MINIMAL -model_unzipped_folder=<HOME>\fwg_model_cache\cmip6_global


2026-10-17 22:12:10 - INFO - Processing generated files in: ./morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2\weather
2026-10-17 22:12:10 - INFO - Moving './morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2\weather\GBR_-_LONDON-GATWICK_Ensemble_ssp245_2050.epw' to './final_results_workflow_global_v5\London_SSP2-4.5_2050_CENTRAL.epw'
2026-10-17 22:12:10 - INFO - Moving './morphing_temp_results\GBR_London.Gatwick.037760_IWEC_uhi_type-2\weather\GBR_-_LONDON-GATWICK_Ensemble_ssp245_2050.stat' to './final_results_workflow_global_v5\London_SSP2-4.5_2050_CENTRAL.stat'
2026-10-17 22:12:10 - INFO - 'GBR_London.Gatwick.037760_IWEC_uhi_type-2_FutureWeatherGenerator-SolarQC_multimodel_constrained_ensemble_perez_1990.epw' is not a future scenario file. It will be left in the temp directory.
2026-10-17 22:12:10 - INFO - 'GBR_London.Gatwick.037760_IWEC_uhi_type-2_FutureWeatherGenerator-SolarQC_multimodel_constrained_ensemble_perez_1990.stat' is not a future scenario file


-------------------- Executing FWG for sevilla_in_this_one_the_uhi_is_type-1.epw --------------------
  Full Command (for reference): java -jar <FWG_JAR_DIR>\FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar -g -epw=D:\Python\pyfwg\pyfwg\tutorials\epws\wo_pattern\sevilla_in_this_one_the_uhi_is_type-1.epw -output_folder=D:\Python\pyfwg\pyfwg\tutorials\morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1\ -output_layout=ORGANIZED -models=BCC_CSM2_MR,CanESM5 -ensemble=true -temp_shift_winter=0.0 -temp_shift_summer=0.0 -smooth_hours=72 -multithread=true -grid_interpolation_method=IDW -epw_dictionary_limits=true -output_type=EPW -scenarios=ssp245 -timeframes=2050 -uncertainty_variant=CENTRAL -output_profile=MINIMAL -model_unzipped_folder=<HOME>\fwg_model_cache\cmip6_global


2026-10-17 22:12:31 - INFO - Processing generated files in: ./morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1\weather
2026-10-17 22:12:31 - INFO - Moving './morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1\weather\ESP_-_SEVILLA_Ensemble_ssp245_2050.epw' to './final_results_workflow_global_v5\Seville_SSP2-4.5_2050_CENTRAL.epw'
2026-10-17 22:12:31 - INFO - Moving './morphing_temp_results\sevilla_in_this_one_the_uhi_is_type-1\weather\ESP_-_SEVILLA_Ensemble_ssp245_2050.stat' to './final_results_workflow_global_v5\Seville_SSP2-4.5_2050_CENTRAL.stat'
2026-10-17 22:12:31 - INFO - 'sevilla_in_this_one_the_uhi_is_type-1_FutureWeatherGenerator-SolarQC_multimodel_constrained_ensemble_perez_1990.epw' is not a future scenario file. It will be left in the temp directory.
2026-10-17 22:12:31 - INFO - 'sevilla_in_this_one_the_uhi_is_type-1_FutureWeatherGenerator-SolarQC_multimodel_constrained_ensemble_perez_1990.stat' is not a future scenario file. It will be left in the temp di

Let's take a look at the files we just generated:

In [6]:
output_files_v5 = [i for i in os.listdir('./final_results_workflow_global_v5')]
output_files_v5

['London_SSP2-4.5_2050_CENTRAL.epw',
 'London_SSP2-4.5_2050_CENTRAL.stat',
 'Seville_SSP2-4.5_2050_CENTRAL.epw',
 'Seville_SSP2-4.5_2050_CENTRAL.stat']

Let's delete the files of this example:

In [7]:
import shutil

if os.path.exists('./final_results_workflow_global_v5'):
    shutil.rmtree('./final_results_workflow_global_v5')

## Using `MorphingWorkflowEurope`

### Europe v2 Example

**FutureWeatherGenerator Europe v2** also adopts the keyword-based CLI. Remember to use `fwg_rcm_pairs` instead of `fwg_gcms`, and `{rcp}` instead of `{ssp}` for renaming.

In [20]:
from pyfwg import MorphingWorkflowEurope, DEFAULT_EUROPE_RCMS

# Europe v2 JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename if you use another version.
jar_path_europe = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator_Europe_v2.0.2.jar')

workflow_eu = MorphingWorkflowEurope()
workflow_eu.map_categories(
    epw_files=epw_files_without_pattern,
    keyword_mapping=mapping_rules
)

workflow_eu.configure_and_preview(
    delete_temp_files=False,
    final_output_dir='./final_results_workflow_europe',
    output_filename_pattern='{city}_{uhi}_{rcp}_{year}',
    scenario_mapping={'rcp26': 'RCP-2.6'},
    fwg_jar_path=jar_path_europe,
    fwg_rcm_pairs=[list(DEFAULT_EUROPE_RCMS)[0]],
    fwg_epw_original_lcz=2,
    fwg_target_uhi_lcz=3,
    fwg_version='2', # Europe v2 keyword CLI
    fwg_output_type='PORTUGAL_CSV' # New: Generate CSV files
)

workflow_eu.execute_morphing()

2025-12-30 12:41:15 - INFO - --- Step 1: Mapping categories from filenames ---
2025-12-30 12:41:15 - INFO - Mapped 'epws/wo_pattern\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw': {'city': 'London', 'uhi': 'type-2'}
2025-12-30 12:41:15 - INFO - Mapped 'epws/wo_pattern\sevilla_in_this_one_the_uhi_is_type-1.epw': {'city': 'Seville', 'uhi': 'type-1'}
2025-12-30 12:41:15 - INFO - Category mapping complete.
2025-12-30 12:41:15 - INFO - --- Step 2: Configuring and Previewing Morphing Plan ---
2025-12-30 12:41:15 - INFO - --- Step 4: Executing morphing workflow ---
2025-12-30 12:41:15 - INFO - Validating LCZ availability for GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw...
2025-12-30 12:41:15 - INFO - Checking LCZ pair (Original: 2, Target: 3) availability for GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw...
2025-12-30 12:41:15 - INFO - --- Applying UHI effect to GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw ---
2025-12-30 12:41:15 - INFO - Executing command: java -jar "<FWG_JAR_DIR>\FutureWe


          MORPHING CONFIGURATION & PREVIEW
  - FWG JAR Path: <FWG_JAR_DIR>\FutureWeatherGenerator_Europe_v2.0.2.jar
  - Final Output Directory: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_europe
  - EPWs to be Morphed (2 files):
    - GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw
    - sevilla_in_this_one_the_uhi_is_type-1.epw

  For input file: GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw
    -> Generated 'rcp26_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_europe\London_type-2_RCP-2.6_2050.epw
    -> Generated 'rcp45_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_europe\London_type-2_rcp45_2050.epw
    -> Generated 'rcp85_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_europe\London_type-2_rcp85_2050.epw
    -> Generated 'rcp26_2080.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\final_results_workflow_europe\London_type-2_RCP-2.6_2080.epw
    -> Generated '

2025-12-30 12:41:32 - INFO - UHI effect applied successfully.
2025-12-30 12:41:32 - INFO - LCZ pair (Original: 2, Target: 3) is available.
2025-12-30 12:41:32 - INFO - Copied input file to temporary directory: ./morphing_temp_results_europe\GBR_London.Gatwick.037760_IWEC_uhi_type-2\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw



-------------------- Executing FWG for GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw --------------------
  Full Command (for reference): java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_Europe_v2.0.2.jar" -epw=D:\Python\pyfwg\pyfwg\tutorials\morphing_temp_results_europe\GBR_London.Gatwick.037760_IWEC_uhi_type-2\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw D:\Python\pyfwg\pyfwg\tutorials\epws\wo_pattern\GBR_London.Gatwick.037760_IWEC_uhi_type-2.epw -models=MOHC_HadGEM2_ES_SMHI_RCA4 -ensemble=true -temp_shift_winter=0.0 -temp_shift_summer=0.0 -smooth_hours=72 -multithread=true -grid_interpolation_method=IDW -solar_correction=By_Month -diffuse_method=Engerer_2015 -uhi=true:2:3 -output_type=PORTUGAL_CSV


2025-12-30 12:42:11 - INFO - Processing generated files in: ./morphing_temp_results_europe\GBR_London.Gatwick.037760_IWEC_uhi_type-2
2025-12-30 12:42:11 - INFO - Skipping auxiliary file: '00_logs'
2025-12-30 12:42:11 - INFO - Skipping auxiliary file: '01_processing_transcripts'
2025-12-30 12:42:11 - INFO - Skipping auxiliary file: '02_morphing_errors_and_warnings'
2025-12-30 12:42:11 - INFO - Skipping auxiliary file: '03_model_variables'
2025-12-30 12:42:11 - INFO - Skipping auxiliary file: '04_epw_comparisons'
2025-12-30 12:42:11 - INFO - Moving './morphing_temp_results_europe\GBR_London.Gatwick.037760_IWEC_uhi_type-2\GBR_-_LONDON-GATWICK_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp26_2050.csv' to './final_results_workflow_europe\London_type-2_RCP-2.6_2050.csv'
2025-12-30 12:42:11 - INFO - Moving './morphing_temp_results_europe\GBR_London.Gatwick.037760_IWEC_uhi_type-2\GBR_-_LONDON-GATWICK_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp26_2050.stat' to './final_results_workflow_europe\Lo


-------------------- Executing FWG for sevilla_in_this_one_the_uhi_is_type-1.epw --------------------
  Full Command (for reference): java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_Europe_v2.0.2.jar" -epw=D:\Python\pyfwg\pyfwg\tutorials\morphing_temp_results_europe\sevilla_in_this_one_the_uhi_is_type-1\sevilla_in_this_one_the_uhi_is_type-1.epw D:\Python\pyfwg\pyfwg\tutorials\epws\wo_pattern\sevilla_in_this_one_the_uhi_is_type-1.epw -models=MOHC_HadGEM2_ES_SMHI_RCA4 -ensemble=true -temp_shift_winter=0.0 -temp_shift_summer=0.0 -smooth_hours=72 -multithread=true -grid_interpolation_method=IDW -solar_correction=By_Month -diffuse_method=Engerer_2015 -uhi=true:2:3 -output_type=PORTUGAL_CSV


2025-12-30 12:43:06 - INFO - Processing generated files in: ./morphing_temp_results_europe\sevilla_in_this_one_the_uhi_is_type-1
2025-12-30 12:43:06 - INFO - Skipping auxiliary file: '00_logs'
2025-12-30 12:43:06 - INFO - Skipping auxiliary file: '01_processing_transcripts'
2025-12-30 12:43:06 - INFO - Skipping auxiliary file: '02_morphing_errors_and_warnings'
2025-12-30 12:43:06 - INFO - Skipping auxiliary file: '03_model_variables'
2025-12-30 12:43:06 - INFO - Skipping auxiliary file: '04_epw_comparisons'
2025-12-30 12:43:06 - INFO - Moving './morphing_temp_results_europe\sevilla_in_this_one_the_uhi_is_type-1\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp26_2050.csv' to './final_results_workflow_europe\Seville_type-1_RCP-2.6_2050.csv'
2025-12-30 12:43:06 - INFO - Moving './morphing_temp_results_europe\sevilla_in_this_one_the_uhi_is_type-1\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp26_2050.stat' to './final_results_workflow_europe\Seville_type-1_RCP-2.6_2050

Let's take a look at the files we just generated:

In [21]:
output_files = [i for i in os.listdir('./final_results_workflow_europe')]
output_files

['London_type-2_RCP-2.6_2050.csv',
 'London_type-2_RCP-2.6_2050.stat',
 'London_type-2_RCP-2.6_2080.csv',
 'London_type-2_RCP-2.6_2080.stat',
 'London_type-2_rcp45_2050.csv',
 'London_type-2_rcp45_2050.stat',
 'London_type-2_rcp45_2080.csv',
 'London_type-2_rcp45_2080.stat',
 'London_type-2_rcp85_2050.csv',
 'London_type-2_rcp85_2050.stat',
 'London_type-2_rcp85_2080.csv',
 'London_type-2_rcp85_2080.stat',
 'Seville_type-1_RCP-2.6_2050.csv',
 'Seville_type-1_RCP-2.6_2050.stat',
 'Seville_type-1_RCP-2.6_2080.csv',
 'Seville_type-1_RCP-2.6_2080.stat',
 'Seville_type-1_rcp45_2050.csv',
 'Seville_type-1_rcp45_2050.stat',
 'Seville_type-1_rcp45_2080.csv',
 'Seville_type-1_rcp45_2080.stat',
 'Seville_type-1_rcp85_2050.csv',
 'Seville_type-1_rcp85_2050.stat',
 'Seville_type-1_rcp85_2080.csv',
 'Seville_type-1_rcp85_2080.stat']

### Europe v3.x Example (FWG Engine)

`MorphingWorkflowEurope` also works with **Europe v3.x** (`FutureWeatherGenerator-CORDEX-CMIP5-Europe-EUR-11-v3.X.X.jar`), which runs on the FWG Engine. The FWG Engine arguments are the same as in the Global v5.x example, but the scenarios are RCPs (e.g., `['rcp45', 'rcp85']`) and the models are GCM-RCM pairs. Both of our example files (London and Seville) are located within the European domain (EUR-11).

> **Note:** The Europe v3 JAR was not available when this notebook was run, so the following cells have not been executed and their output is not shown.

In [ ]:
# Europe v3 JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename if you use another version.
jar_path_europe_v3 = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator-CORDEX-CMIP5-Europe-EUR-11-v3.2.0.jar')

# Use a separate climate-data cache for each distribution of the tool.
model_cache_dir_europe = os.path.join(os.path.expanduser('~'), 'fwg_model_cache', 'cordex_eur11')

workflow_eu_v3 = MorphingWorkflowEurope()
workflow_eu_v3.map_categories(
    epw_files=epw_files_without_pattern,
    keyword_mapping=mapping_rules
)

workflow_eu_v3.configure_and_preview(
    final_output_dir='./final_results_workflow_europe_v3',
    output_filename_pattern='{city}_{rcp}_{year}',
    scenario_mapping={'rcp45': 'RCP-4.5', 'rcp85': 'RCP-8.5'},
    fwg_jar_path=jar_path_europe_v3,
    fwg_rcm_pairs=['ICHEC_EC_EARTH_SMHI_RCA4', 'MPI_M_MPI_ESM_LR_SMHI_RCA4'],  # Two pairs to create the ensemble
    fwg_scenarios=['rcp45', 'rcp85'],
    fwg_timeframes=[2050],
    fwg_model_unzipped_folder=model_cache_dir_europe
)

workflow_eu_v3.execute_morphing()

Let's take a look at the files we just generated, and then delete them:

In [ ]:
print(os.listdir('./final_results_workflow_europe_v3'))

In [ ]:
import shutil

if os.path.exists('./final_results_workflow_europe_v3'):
    shutil.rmtree('./final_results_workflow_europe_v3')

## Using `MorphingWorkflowCordex`

The **CORDEX regional distributions** of the Future Weather Generator (Middle East and North Africa `MNA-22`, Africa `AFR-22`, North America `NAM-44`, Central America `CAM-44`, South America `SAM-44`, West & South Asia `WAS-22`, Southeast Asia `SEA-22` and Australasia `AUS-22`) run on the FWG Engine and are handled by the `MorphingWorkflowCordex` class. It works like `MorphingWorkflowEurope` (`fwg_rcm_pairs` and the `{rcp}` placeholder), with two differences:

*   **Region detection**: The region is detected from the JAR (from its packaged manifest or, otherwise, from its filename), and only its scenarios and GCM-RCM pairs are accepted. They are listed in `CORDEX_REGIONS`, and the preview only includes the scenarios available in the region.
*   **No UHI**: These distributions do not support the Urban Heat Island effect, so the class has no UHI or LCZ arguments.

The weather files must be located within the domain of the distribution. In this example, we morph the Seville file with the Middle East and North Africa distribution (`MNA-22`), which only provides the `rcp85` scenario. The London file is outside this domain, so we leave it out. For other locations, use the JAR of their region (e.g., `NAM-44` for North America). We also use a separate climate-data cache for this distribution, since some GCM-RCM pairs (e.g., `ICHEC_EC_EARTH_SMHI_RCA4`) are provided by several regions.

> **Note:** The CORDEX regional JARs were not available when this notebook was run, so the following cells have not been executed and their output is not shown.

In [ ]:
from pyfwg import MorphingWorkflowCordex, CORDEX_REGIONS

# CORDEX regional JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename for another region or version.
jar_path_mna = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator-CORDEX-CMIP5-Middle-East-and-North-Africa-MNA-22-v1.2.0.jar')
model_cache_dir_mna = os.path.join(os.path.expanduser('~'), 'fwg_model_cache', 'cordex_mna22')

print(CORDEX_REGIONS['MNA-22'])  # Scenarios and GCM-RCM pairs of the region

workflow_cordex = MorphingWorkflowCordex()
workflow_cordex.map_categories(
    epw_files=[f for f in epw_files_without_pattern if 'sevilla' in f],  # Only the file within the MNA-22 domain
    keyword_mapping=mapping_rules
)

workflow_cordex.configure_and_preview(
    final_output_dir='./final_results_workflow_cordex',
    output_filename_pattern='{city}_{rcp}_{year}',
    scenario_mapping={'rcp85': 'RCP-8.5'},
    fwg_jar_path=jar_path_mna,
    fwg_rcm_pairs=sorted(CORDEX_REGIONS['MNA-22']['rcm_pairs']),  # Both GCM-RCM pairs of the region
    fwg_timeframes=[2050],
    fwg_model_unzipped_folder=model_cache_dir_mna
)

In [ ]:
print(f"CORDEX domain detected from the JAR: {workflow_cordex.cordex_domain}")

workflow_cordex.execute_morphing()

Let's take a look at the files we just generated, and then delete them together with the temporary folder of this example:

In [ ]:
print(os.listdir('./final_results_workflow_cordex'))

In [ ]:
import shutil

for folder in ['./final_results_workflow_cordex', './morphing_temp_results_cordex']:
    if os.path.exists(folder):
        shutil.rmtree(folder)

## Cleaning up

Once the workflow is finished, you can delete the temporary folders created during the process if you don't need them anymore. If you did not enable `delete_temp_files=True` in `configure_and_preview`, you can do it manually.

**Note on Analysis Files:** In FutureWeatherGenerator v4 (Global) and v2 (Europe), the tool generates several analysis folders (e.g., `00_logs`, `01_processing_transcripts`, etc.) when enabled. These folders are **not** moved to the final output directory; they remain in the temporary directory for inspection. If you delete the temporary directory, these files will be lost.

**Note on FWG Engine Files:** With the FWG Engine (Global v5.x, Europe v3.x and CORDEX regional), the tool writes the weather files to the `weather` subfolder of the temporary directory, and its reports and manifests to other subfolders (e.g., `reports` and `metadata`). Only the weather and `.stat` files of the future scenarios are moved to the final output directory; the rest remain in the temporary directory.

In [22]:
import shutil
import os

# Clean up the results folders
for folder in ['./morphing_temp_results', './morphing_temp_results_europe', './final_results_workflow_global', './final_results_workflow_europe', './00_logs']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Deleted: {folder}")

Deleted: ./morphing_temp_results
Deleted: ./morphing_temp_results_europe
Deleted: ./final_results_workflow_global
Deleted: ./final_results_workflow_europe
